In [56]:
import os
import json
import sqlite3
import random
import string
from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
google_api_key = os.getenv("GEMINI_API_KEY")
print("Gemini API Key set" if google_api_key else "Gemini API Key not set")

gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
MODEL = "gemini-3.5-flash-lite"
DB = "airline.db"


Gemini API Key set


In [57]:
today = date.today()
system_message = f"""
You are the assistant for the airline FlightAI. Today is {today:%A, %d %B %Y} ({today.isoformat()}).
You can search flights, book flights, look up or cancel bookings, answer policy questions, and raise support tickets.

Rules:
- Always use tools for flights, prices, bookings and policies. Never invent flights, prices, PNRs or policies.
- Convert relative dates like "tomorrow" to YYYY-MM-DD using today's date.
- Show flights as a short markdown table: Flight ID, Departure, Arrival, Duration, Price (INR). Always include the Flight ID.
- Before calling create_booking you MUST: have the flight_id, the passenger's full name and email,
  show a summary with the total price, and get an explicit "yes" from the user. Never book without it.
- Before cancelling, confirm the PNR with the user.
- If you can't help, offer to create a support ticket.
- Be concise and friendly.
"""


In [58]:
def get_conn():
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row   # lets us do dict(row)
    return conn

def resolve_airport(conn, place):
    row = conn.execute("SELECT code FROM airports WHERE code = ? OR LOWER(city) = LOWER(?)",
                       (place.strip().upper(), place.strip())).fetchone()
    return row["code"] if row else None

def generate_code(length=6):
    return "".join(random.choices(string.ascii_uppercase + string.digits, k=length))

TIME_WINDOWS = {"morning": ("05:00", "12:00"), "afternoon": ("12:00", "17:00"),
                "evening": ("17:00", "21:00"), "night": ("21:00", "23:59")}


In [59]:
def search_flights(origin, destination, date, time_of_day="any", sort_by="price", max_results=5):
    print(f"TOOL: search_flights {origin} -> {destination} on {date} ({time_of_day})", flush=True)
    with get_conn() as conn:
        o, d = resolve_airport(conn, origin), resolve_airport(conn, destination)
        if not o or not d:
            cities = [r["city"] for r in conn.execute("SELECT city FROM airports")]
            return {"error": f"Unknown city. We fly to: {', '.join(cities)}"}
        start, end = TIME_WINDOWS.get(time_of_day, ("00:00", "23:59"))
        order = {"price": "price", "departure": "departure", "duration": "duration_min"}.get(sort_by, "price")
        rows = conn.execute(f"""
            SELECT flight_id, flight_no, origin, destination, departure, arrival, duration_min, price, seats_available
            FROM flights
            WHERE origin = ? AND destination = ? AND DATE(departure) = ?
              AND TIME(departure) >= TIME(?) AND TIME(departure) < TIME(?) AND seats_available > 0
            ORDER BY {order} LIMIT ?""", (o, d, date, start, end, max_results)).fetchall()
    return {"flights": [dict(r) for r in rows]} if rows else {"flights": [], "message": "No flights found"}


In [60]:
def create_booking(flight_id, passenger_name, passenger_email, seats=1):
    print(f"TOOL: create_booking {flight_id} x{seats}", flush=True)
    if not 1 <= seats <= 9:
        return {"error": "Seats must be between 1 and 9"}
    if "@" not in passenger_email:
        return {"error": "Invalid email address"}
    with get_conn() as conn:   # commits on success, rolls back on error
        flight = conn.execute("SELECT * FROM flights WHERE flight_id = ?", (flight_id,)).fetchone()
        if not flight:
            return {"error": "Flight not found"}
        updated = conn.execute(
            "UPDATE flights SET seats_available = seats_available - ? WHERE flight_id = ? AND seats_available >= ?",
            (seats, flight_id, seats))
        if updated.rowcount == 0:
            return {"error": "Not enough seats available"}
        pnr = generate_code()
        ticket_no = "098" + "".join(random.choices(string.digits, k=10))
        total = flight["price"] * seats
        conn.execute("INSERT INTO bookings VALUES (?,?,?,?,?,?,?,?,?)",
                     (pnr, flight_id, passenger_name, passenger_email, seats, total, "CONFIRMED",
                      ticket_no, datetime.now().isoformat(timespec="seconds")))
    return {"status": "CONFIRMED", "pnr": pnr, "ticket_no": ticket_no, "passenger_name": passenger_name,
            "seats": seats, "total_price": total, "flight": dict(flight)}


In [61]:
from datetime import timedelta
search_flights("Delhi", "Mumbai", (date.today() + timedelta(days=1)).isoformat(), "morning")

TOOL: search_flights Delhi -> Mumbai on 2026-10-27 (morning)


{'flights': [{'flight_id': 'FA102-20261027',
   'flight_no': 'FA102',
   'origin': 'DEL',
   'destination': 'BOM',
   'departure': '2026-10-27T08:30',
   'arrival': '2026-10-27T10:30',
   'duration_min': 120,
   'price': 4450.0,
   'seats_available': 6},
  {'flight_id': 'FA101-20261027',
   'flight_no': 'FA101',
   'origin': 'DEL',
   'destination': 'BOM',
   'departure': '2026-10-27T06:00',
   'arrival': '2026-10-27T08:10',
   'duration_min': 130,
   'price': 4860.0,
   'seats_available': 6},
  {'flight_id': 'FA103-20261027',
   'flight_no': 'FA103',
   'origin': 'DEL',
   'destination': 'BOM',
   'departure': '2026-10-27T11:15',
   'arrival': '2026-10-27T13:45',
   'duration_min': 150,
   'price': 5030.0,
   'seats_available': 40}]}

In [62]:
search_flights_function = {
    "name": "search_flights",
    "description": "Search available flights between two cities on a date. Use for any flight, price or schedule question.",
    "parameters": {
        "type": "object",
        "properties": {
            "origin": {"type": "string", "description": "Departure city name or airport code, e.g. Delhi or DEL"},
            "destination": {"type": "string", "description": "Arrival city name or airport code"},
            "date": {"type": "string", "description": "Travel date in YYYY-MM-DD"},
            "time_of_day": {"type": "string", "enum": ["morning", "afternoon", "evening", "night", "any"]},
            "sort_by": {"type": "string", "enum": ["price", "departure", "duration"]},
        },
        "required": ["origin", "destination", "date"],
        "additionalProperties": False,
    },
}

create_booking_function = {
    "name": "create_booking",
    "description": "Book a flight. Only call AFTER the user has explicitly confirmed the flight, passenger details and total price.",
    "parameters": {
        "type": "object",
        "properties": {
            "flight_id": {"type": "string", "description": "flight_id from search_flights, e.g. FA101-20261024"},
            "passenger_name": {"type": "string", "description": "Passenger full name"},
            "passenger_email": {"type": "string", "description": "Email for the confirmation"},
            "seats": {"type": "integer", "description": "Number of seats, default 1"},
        },
        "required": ["flight_id", "passenger_name", "passenger_email"],
        "additionalProperties": False,
    },
}


In [63]:
tools = [{"type": "function", "function": f} for f in [
    search_flights_function, create_booking_function]]


In [64]:
TOOL_FUNCTIONS = {
    "search_flights": search_flights, "create_booking": create_booking
}

def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        try:
            arguments = json.loads(tool_call.function.arguments or "{}")
            if tool_call.function.name == "search_flights":
                city = arguments.get('destination')
                cities.append(city)
            result = TOOL_FUNCTIONS[tool_call.function.name](**arguments)
        except Exception as e:
            result = {"error": f"{type(e).__name__}: {e}"}
        responses.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return responses, cities

In [65]:
import base64
from io import BytesIO
from PIL import Image

In [66]:
def artist(city):
    image_response = gemini.images.generate(
            model="gemini-2.5-flash-image",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [67]:
from google import genai
from google.genai import types
import wave

client = genai.Client(api_key=google_api_key)  # load from env var, don't hardcode

def talker(message, filename="output.wav"):
    response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=message,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
                )
            ),
        ),
    )
    pcm = response.candidates[0].content.parts[0].inline_data.data

    # Gemini returns raw PCM (24 kHz, 16-bit, mono), so wrap it as WAV
    with wave.open(filename, "wb") as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(24000)
        f.writeframes(pcm)
    return filename

In [68]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image

In [69]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)   # Gradio 6 removed type="messages"; messages is now the only format
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=False, auth=("admin", "admin"))

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


TOOL: search_flights Delhi -> Mumbai on 2026-10-26 (any)
